# Моделирование мира маргариток: Динамика числа популяций

**Цель:** Исследовать динамику численности черных и белых маргариток в модели Daisyworld при фиксированной солнечной активности.
Модель демонстрирует конкуренцию между видами с разным альбедо: черные маргаритки (низкое альбедо) нагревают почву,
белые (высокое альбедо) — охлаждают. Это создает отрицательную обратную связь, стабилизирующую климат планеты.

## Инициализация проекта
Активируем окружение проекта DrWatson и загружаем необходимые библиотеки

In [1]:
using DrWatson
@quickactivate "project"
using Agents
using DataFrames
using Plots

## Подключение модуля с моделью
В файле `src/daisyworld.jl` содержится определение самой модели:
- Типы агентов и их свойства (цвет, возраст, альбедо).
- Функции для инициализации мира, правила роста и размножения.
- Параметры модели (солнечная постоянная, альбедо маргариток, диффузия тепла и так далее)

In [2]:
include(srcdir("daisyworld.jl"))

daisyworld (generic function with 1 method)

Используем CairoMakie для расширенной визуализации (тепловые карты, сложные графики)

In [3]:
using CairoMakie

## Определение агрегирующих функций для сбора данных

Для отслеживания динамики популяции определим функции, которые подсчитывают количество черных и белых маргариток в модели
- `black(a)`: возвращает `true`, если маргаритка черная
- `white(a)`: возвращает `true`, если маргаритка белая
- `adata`: список агрегаций, которые будут собираться на каждом шаге (функция и применяемая к ней операция)

In [4]:
black(a) = a.breed == :black
white(a) = a.breed == :white
adata = [(black, count), (white, count)]

2-element Vector{Tuple{Function, typeof(count)}}:
 (Main.var"##278".black, count)
 (Main.var"##278".white, count)

## Инициализация модели

Создаем модель Daisyworld с начальной солнечной светимостью 1.0. Остальные параметры по умолчанию:
- griddims = (30, 30) — размер мира
- max_age = 25 — максимальный возраст маргаритки
- init_white = 0.2, init_black = 0.2 — начальное покрытие 20% белыми и 20% черными
- albedo_white = 0.75, albedo_black = 0.25 — альбедо видов

In [5]:
model = daisyworld(; solar_luminosity = 1.0)

StandardABM with 360 agents of type Daisy
 agents container: Dict
 space: GridSpaceSingle with size (30, 30), metric=chebyshev, periodic=true
 scheduler: fastest
 properties: temperature, solar_luminosity, max_age, surface_albedo, ratio, solar_change, tick, scenario

## Запускаем модель и сбор данных
Запускаем модель на 1000 шагов (достаточно для выхода на стационарный режим)
- `run!`: выполняет симуляцию и собирает данные в соответствии с `adata`
- `agent_df`: DataFrame с данными об агентах на каждом шаге (численность черных и белых)
- `model_df`: DataFrame с данными модели (температура, светимость и др.)

In [6]:
agent_df, model_df = run!(model, 1000; adata)

(1001×3 DataFrame
  Row │ time   count_black  count_white 
      │ Int64  Int64        Int64       
──────┼─────────────────────────────────
    1 │     0          180          180
    2 │     1          416          168
    3 │     2          560          249
    4 │     3          581          295
    5 │     4          583          313
    6 │     5          577          321
    7 │     6          572          326
    8 │     7          567          330
  ⋮   │   ⋮         ⋮            ⋮
  995 │   994          542          354
  996 │   995          544          354
  997 │   996          537          363
  998 │   997          527          372
  999 │   998          522          376
 1000 │   999          522          376
 1001 │  1000          520          376
                        986 rows omitted, 0×0 DataFrame)

## Визуализация динамики популяции

Строим график изменения численности черных и белых маргариток во времени.
На графике видно, как виды конкурируют за пространство и достигают равновесия.

In [7]:
figure = Figure(size = (600, 400));

`xlabel`: время, шаги (1 шаг = 1 единица времени модели)
`ylabel`: количество маргариток (максимум ~900 при полном покрытии мира)

In [8]:
ax = figure[1, 1] = Axis(figure, xlabel = "tick", ylabel = "daisy count")

Makie.Axis with 0 plots:


график для черных маргариток (низкое альбедо, нагревают планету)

In [9]:
blackl = lines!(ax, agent_df[!, :time], agent_df[!, :count_black], color = :black)

Makie.Lines{Tuple{Vector{GeometryBasics.Point{2, Float64}}}}

график для белых маргариток (высокое альбедо, охлаждают планету)

In [10]:
whitel = lines!(ax, agent_df[!, :time], agent_df[!, :count_white], color = :orange)
Legend(figure[1, 2], [blackl, whitel], ["black", "white"], labelsize = 12)

Makie.Legend()

Сохраняем график в папку `plots/`

In [11]:
save(plotsdir("daisy_count.png"), figure)